# LSN-003｜冻结神经元语义
**对应：RMD-003 · PFR3/PFR4 · PDP3/PDP4**

## 本课问题
如果 Python、AI 和未来的 RTL 对“同一个 LIF”有不同理解，哪一个才算正确？

> 本课唯一主要新概念：**实现之前必须先冻结可测试的语义。**

## 一个很小但致命的歧义
当更新后的膜电位恰好等于 threshold 时，应该 spike 吗？

`>=` 和 `>` 看起来只差一个字符，但它们定义的是两个不同模型。

In [ ]:
def step_ge(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v >= threshold
    return (reset if spike else new_v), spike

def step_gt(v, current, alpha=1.0, threshold=1.0, reset=0.0):
    new_v = alpha * v + current
    spike = new_v > threshold
    return (reset if spike else new_v), spike

v0, current = 0.75, 0.25
print('>= rule:', step_ge(v0, current))
print(' > rule:', step_gt(v0, current))

## Observe
同样的初始状态和输入，因为一个符号不同，spike sequence 已经可能分叉。

还需要明确的语义包括：
- decay 与 input 的先后顺序；
- threshold 比较发生在什么时候；
- spike 后 reset 到多少；
- refractory 期间是否仍然积分输入；
- fixed-point rounding/saturation 在哪一步发生；
- 一个 step 内多个输入怎样累积。

In [ ]:
semantic_decisions = {
    'threshold_comparison': 'TBD: >= or >',
    'update_order': 'TBD',
    'reset_value': 'TBD',
    'refractory_rule': 'TBD',
    'rounding_rule': 'TBD',
    'overflow_rule': 'TBD',
}

for key, value in semantic_decisions.items():
    print(f'{key:24s} {value}')

## AI Task
让 AI 列出一份“实现 LIF 前必须明确的歧义清单”，并要求它为每个歧义给出一个能区分两种语义的最小测试向量。

## Human Check
- 为什么“常见 LIF 写法”不能替代我们的 spec？
- test oracle 应该引用代码实现，还是引用冻结后的语义？
- 如果 RTL 与 Python 不一致，哪一层有权决定谁错了？

## Engineering Handoff
本课的产物不是另一段算法代码，而是一次 spec checkpoint：更新 MDD 的数值/接口语义、TDD 的 oracle，以及 TRACE 的对应关系。

## Exit Ticket
`semantic_decisions` 中不再有影响实现的 TBD；每项关键决策都有至少一个能失败的测试例。完成后才进入 LSN-004 / RMD-003A。